In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — Install / check dependencies
# ═══════════════════════════════════════════════════════════════════════════════
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — Upload your corpus
# ═══════════════════════════════════════════════════════════════════════════════
from google.colab import files
uploaded = files.upload()   # select corpus_small_clean.txt
CORPUS_PATH = list(uploaded.keys())[0]
print(f"Corpus uploaded: {CORPUS_PATH}")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — Full model + dataset + training code (all-in-one)
# ═══════════════════════════════════════════════════════════════════════════════
import os, math, time, csv
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from torch.utils.data import Dataset, DataLoader

# ──────────────────────────────────────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────────────────────────────────────

@dataclass
class GPTConfig:
    vocab_size       : int   = None   # set automatically from corpus
    max_len          : int   = 256
    d_model          : int   = 256
    n_heads          : int   = 8
    n_layers         : int   = 6
    dropout          : float = 0.1
    attn_backend     : str   = "flash"    # "flash" | "standard"
    pos_emb_type     : str   = "learned"  # "learned" | "sinusoidal" | "rope"
    batch_size       : int   = 64
    lr               : float = 3e-4
    weight_decay     : float = 0.1
    grad_clip        : float = 1.0
    max_iters        : int   = 20_000
    warmup_iters     : int   = 500
    grad_accum_steps : int   = 1
    eval_interval    : int   = 200
    eval_iters       : int   = 50
    sample_interval  : int   = 2000
    sample_length    : int   = 300
    checkpoint_dir   : str   = "checkpoints"
    checkpoint_interval : int = 2000
    train_frac       : float = 0.9

    @property
    def effective_batch_size(self):
        return self.batch_size * self.grad_accum_steps

# ──────────────────────────────────────────────────────────────────────────────
# Tokeniser & Dataset
# ──────────────────────────────────────────────────────────────────────────────

class CharTokenizer:
    def __init__(self, text):
        self.vocab      = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi       = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos       = {i: ch for i, ch in enumerate(self.vocab)}

    def encode(self, text):
        return [self.stoi[ch] for ch in text]

    def decode(self, ids):
        return "".join(self.itos[i] for i in ids)


class PoetryDataset(Dataset):
    def __init__(self, data, block_size):
        self.data       = data
        self.block_size = block_size

    def __len__(self):
        return len(self.data) - self.block_size   # correct: count of valid indices

    def __getitem__(self, idx):
        chunk = self.data[idx : idx + self.block_size + 1]
        return chunk[:-1].clone(), chunk[1:].clone()


def make_datasets(corpus_path, block_size, train_frac=0.9):
    with open(corpus_path, "r", encoding="utf-8") as f:
        text = f.read()
    tokenizer = CharTokenizer(text)
    data      = torch.tensor(tokenizer.encode(text), dtype=torch.long)
    n         = int(len(data) * train_frac)
    train_ds  = PoetryDataset(data[:n], block_size)
    val_ds    = PoetryDataset(data[n:], block_size)
    print(f"Corpus  : {len(data):,} chars | vocab: {tokenizer.vocab_size}")
    print(f"Train   : {len(data[:n]):,} chars ({len(train_ds):,} samples)")
    print(f"Val     : {len(data[n:]):,} chars ({len(val_ds):,} samples)")
    return train_ds, val_ds, tokenizer

# ──────────────────────────────────────────────────────────────────────────────
# Positional Embeddings
# ──────────────────────────────────────────────────────────────────────────────

class LearnedPositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        positions = torch.arange(x.size(1), device=x.device)
        return x + self.embedding(positions)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model, dropout=0.0):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe           = torch.zeros(max_len, d_model)
        position     = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
        div_term     = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                                 * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term[:d_model // 2])
        self.register_buffer("pe", pe)

    def forward(self, x):
        return self.dropout(x + self.pe[: x.size(1)])


class RotaryPositionalEmbedding(nn.Module):
    def __init__(self, head_dim, max_len=4096):
        super().__init__()
        inv_freq = 1.0 / (10000.0 ** (
            torch.arange(0, head_dim, 2, dtype=torch.float) / head_dim))
        self.register_buffer("inv_freq", inv_freq)
        t   = torch.arange(max_len, dtype=inv_freq.dtype)
        emb = torch.cat([torch.outer(t, inv_freq)] * 2, dim=-1)
        self.register_buffer("cos_cached", emb.cos()[None, None])
        self.register_buffer("sin_cached", emb.sin()[None, None])

    @staticmethod
    def _rotate_half(x):
        x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
        return torch.cat([-x2, x1], dim=-1)

    def apply(self, q, k, seq_len):
        cos = self.cos_cached[:, :, :seq_len]
        sin = self.sin_cached[:, :, :seq_len]
        return (q * cos + self._rotate_half(q) * sin,
                k * cos + self._rotate_half(k) * sin)

# ──────────────────────────────────────────────────────────────────────────────
# Attention
# ──────────────────────────────────────────────────────────────────────────────

class MultiHeadCausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout, attn_backend, pos_emb_type):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads      = n_heads
        self.head_dim     = d_model // n_heads
        self.attn_backend = attn_backend
        self.dropout_p    = dropout
        self.qkv_proj     = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj     = nn.Linear(d_model, d_model,     bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.rope         = (RotaryPositionalEmbedding(self.head_dim)
                             if pos_emb_type == "rope" else None)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv_proj(x).split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if self.rope is not None:
            q, k = self.rope.apply(q, k, T)

        if self.attn_backend == "flash":
            out = F.scaled_dot_product_attention(
                q, k, v,
                dropout_p=self.dropout_p if self.training else 0.0,
                is_causal=True,
            )
        else:
            scale  = 1.0 / math.sqrt(self.head_dim)
            scores = (q @ k.transpose(-2, -1)) * scale
            mask   = torch.ones(T, T, device=x.device, dtype=torch.bool).tril()
            scores = scores.masked_fill(~mask, float("-inf"))
            out    = self.attn_dropout(F.softmax(scores, dim=-1)) @ v

        return self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C))

# ──────────────────────────────────────────────────────────────────────────────
# Feed-Forward & Transformer Block
# ──────────────────────────────────────────────────────────────────────────────

class FeedForward(nn.Module):
    def __init__(self, d_model, expansion=4, dropout=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, expansion * d_model, bias=False),
            nn.GELU(),
            nn.Linear(expansion * d_model, d_model, bias=False),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout, attn_backend, pos_emb_type):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadCausalSelfAttention(
            d_model, n_heads, dropout, attn_backend, pos_emb_type)
        self.ln2  = nn.LayerNorm(d_model)
        self.ff   = FeedForward(d_model, dropout=dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

# ──────────────────────────────────────────────────────────────────────────────
# GPT Decoder
# ──────────────────────────────────────────────────────────────────────────────

class GPTDecoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config    = config
        self.token_emb = nn.Embedding(config.vocab_size, config.d_model)

        if config.pos_emb_type == "learned":
            self.pos_emb = LearnedPositionalEmbedding(config.max_len, config.d_model)
        elif config.pos_emb_type == "sinusoidal":
            self.pos_emb = SinusoidalPositionalEncoding(
                config.max_len, config.d_model, config.dropout)
        else:
            self.pos_emb = None   # RoPE is handled inside attention

        self.blocks = nn.ModuleList([
            TransformerBlock(config.d_model, config.n_heads, config.dropout,
                             config.attn_backend, config.pos_emb_type)
            for _ in range(config.n_layers)
        ])
        self.ln_f    = nn.LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight   # weight tying

        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("out_proj.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * config.n_layers))

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, 0.0, 0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None):
        x      = self.token_emb(idx)
        if self.pos_emb is not None:
            x  = self.pos_emb(x)
        for block in self.blocks:
            x  = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss   = (F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
                  if targets is not None else None)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond    = idx[:, -self.config.max_len:]
            logits, _   = self(idx_cond)
            logits      = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _    = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            idx = torch.cat([idx, torch.multinomial(
                F.softmax(logits, dim=-1), num_samples=1)], dim=1)
        return idx

    def num_parameters(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

# ──────────────────────────────────────────────────────────────────────────────
# LR schedule & helpers
# ──────────────────────────────────────────────────────────────────────────────

LOG2E = math.log2(math.e)

def nats_to_bpc(loss):
    return loss * LOG2E

def get_lr(step, config):
    if step < config.warmup_iters:
        return config.lr * step / max(1, config.warmup_iters)
    progress = (step - config.warmup_iters) / max(
        1, config.max_iters - config.warmup_iters)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config.lr / 10 + (config.lr - config.lr / 10) * cosine

@torch.no_grad()
def estimate_metrics(model, loaders, eval_iters, device):
    model.eval()
    out = {}
    for split, loader in loaders.items():
        total, it = 0.0, iter(loader)
        for _ in range(eval_iters):
            try:    x, y = next(it)
            except: it = iter(loader); x, y = next(it)
            _, loss = model(x.to(device), y.to(device))
            total  += loss.item()
        avg                  = total / eval_iters
        out[f"{split}_loss"] = avg
        out[f"{split}_bpc"]  = nats_to_bpc(avg)
    model.train()
    return out

def generate_sample(model, tokenizer, device, n_chars=300,
                    temperature=0.8, top_k=40):
    model.eval()
    ctx = torch.tensor(tokenizer.encode("\n"), dtype=torch.long,
                       device=device).unsqueeze(0)
    with torch.no_grad():
        out = model.generate(ctx, n_chars, temperature, top_k)
    model.train()
    return tokenizer.decode(out[0].tolist())

def unwrap(model):
    """Return the raw GPTDecoder, stripping torch.compile wrapper if present."""
    return model._orig_mod if hasattr(model, "_orig_mod") else model

def save_checkpoint(model, step, config, tokenizer, path):
    """Always saves raw (uncompiled) state dict so loading never breaks."""
    torch.save({
        "step"            : step,
        "model_state"     : unwrap(model).state_dict(),
        "config"          : config,
        "tokenizer_vocab" : tokenizer.vocab,
    }, path)

print("All classes and functions defined ✓")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — Build datasets and model
# ═══════════════════════════════════════════════════════════════════════════════

config = GPTConfig()   # all defaults

train_ds, val_ds, tokenizer = make_datasets(
    CORPUS_PATH, config.max_len, config.train_frac
)
config.vocab_size = tokenizer.vocab_size

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = GPTDecoder(config).to(device)

# torch.compile: free ~10-30% speedup on PyTorch 2.0+
if hasattr(torch, "compile"):
    model = torch.compile(model)

print(f"\nDevice     : {device}")
print(f"Parameters : {unwrap(model).num_parameters():,}")
print(f"Random BPC : {math.log2(config.vocab_size):.2f}  (baseline to beat)")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — Training loop (with early stopping)
# ═══════════════════════════════════════════════════════════════════════════════

train_loader = DataLoader(train_ds, batch_size=config.batch_size,
                          shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=config.batch_size,
                          shuffle=False, drop_last=True)
loaders      = {"train": train_loader, "val": val_loader}

decay_params    = [p for n, p in model.named_parameters()
                   if p.dim() >= 2 and p.requires_grad]
no_decay_params = [p for n, p in model.named_parameters()
                   if p.dim() < 2  and p.requires_grad]
optimizer = torch.optim.AdamW(
    [{"params": decay_params,    "weight_decay": config.weight_decay},
     {"params": no_decay_params, "weight_decay": 0.0}],
    lr=config.lr, betas=(0.9, 0.95),
)

os.makedirs(config.checkpoint_dir, exist_ok=True)
log_path     = os.path.join(config.checkpoint_dir, "training_log.csv")
samples_path = os.path.join(config.checkpoint_dir, "samples.txt")

with open(log_path, "w", newline="") as f:
    csv.writer(f).writerow(
        ["step", "train_loss", "train_bpc", "val_loss", "val_bpc", "lr", "elapsed_s"]
    )

# ── Early stopping state ──────────────────────────────────────────────────────
best_val_loss = float("inf")
patience      = 15      # stop after 15 evals with no improvement
                        # = 15 × 200 steps = 3000 steps without improvement
no_improve    = 0

train_iter = iter(train_loader)
t_start    = time.time()

model.train()
optimizer.zero_grad(set_to_none=True)

for step in range(config.max_iters):

    # ── LR schedule ───────────────────────────────────────────────────────────
    lr = get_lr(step, config)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    # ── Gradient accumulation ─────────────────────────────────────────────────
    for _ in range(config.grad_accum_steps):
        try:    x, y = next(train_iter)
        except: train_iter = iter(train_loader); x, y = next(train_iter)
        x, y    = x.to(device), y.to(device)
        _, loss = model(x, y)
        (loss / config.grad_accum_steps).backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
    optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    # ── Eval & log ────────────────────────────────────────────────────────────
    if step % config.eval_interval == 0 or step == config.max_iters - 1:
        m       = estimate_metrics(model, loaders, config.eval_iters, device)
        elapsed = time.time() - t_start

        print(f"step {step:5d} | "
              f"train {m['train_loss']:.4f} ({m['train_bpc']:.3f} bpc) | "
              f"val {m['val_loss']:.4f} ({m['val_bpc']:.3f} bpc) | "
              f"lr {lr:.2e} | {elapsed:.0f}s  "
              f"[patience {no_improve}/{patience}]")

        with open(log_path, "a", newline="") as f:
            csv.writer(f).writerow([
                step, f"{m['train_loss']:.6f}", f"{m['train_bpc']:.6f}",
                f"{m['val_loss']:.6f}", f"{m['val_bpc']:.6f}",
                f"{lr:.8f}", f"{elapsed:.1f}"])

        # ── Early stopping logic ──────────────────────────────────────────────
        if m["val_loss"] < best_val_loss:
            best_val_loss = m["val_loss"]
            no_improve    = 0
            save_checkpoint(model, step, config, tokenizer,
                            os.path.join(config.checkpoint_dir, "best.pt"))
            print(f"          ✓ New best val loss — checkpoint saved")
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"\n⚠ Early stopping at step {step}.")
                print(f"  Val loss has not improved for {patience} evaluations "
                      f"({patience * config.eval_interval} steps).")
                print(f"  Best val loss: {best_val_loss:.4f} "
                      f"({nats_to_bpc(best_val_loss):.3f} bpc)")
                break

    # ── Sample generation ─────────────────────────────────────────────────────
    if step % config.sample_interval == 0 and step > 0:
        sample = generate_sample(model, tokenizer, device,
                                 n_chars=config.sample_length)
        header = f"\n{'─'*55}\nStep {step}\n{'─'*55}\n"
        print(header + sample)
        with open(samples_path, "a", encoding="utf-8") as f:
            f.write(header + sample + "\n")

    # ── Periodic checkpoint ───────────────────────────────────────────────────
    if step > 0 and step % config.checkpoint_interval == 0:
        save_checkpoint(model, step, config, tokenizer,
                        os.path.join(config.checkpoint_dir, f"step_{step}.pt"))

print(f"\nDone. Best val: {best_val_loss:.4f} ({nats_to_bpc(best_val_loss):.3f} bpc)")

# ── Restore best model for generation (load into unwrapped model) ─────────────
ckpt = torch.load(os.path.join(config.checkpoint_dir, "best.pt"), weights_only=False)
unwrap(model).load_state_dict(ckpt["model_state"])
print(f"Best model restored from step {ckpt['step']}")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6 — Plot training curves
# ═══════════════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

steps, tr_loss, vl_loss, tr_bpc, vl_bpc, lrs = [], [], [], [], [], []
with open(log_path, newline="") as f:
    for row in csv.DictReader(f):
        steps.append(int(row["step"]))
        tr_loss.append(float(row["train_loss"]))
        vl_loss.append(float(row["val_loss"]))
        tr_bpc.append(float(row["train_bpc"]))
        vl_bpc.append(float(row["val_bpc"]))
        lrs.append(float(row["lr"]))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Training curves — Character-level GPT on poetry corpus", fontsize=13)

ax = axes[0]
ax.plot(steps, tr_loss, label="train")
ax.plot(steps, vl_loss, "--", label="val")
ax.set(xlabel="Step", ylabel="Loss (nats)", title="Cross-entropy loss")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(steps, tr_bpc, label="train")
ax.plot(steps, vl_bpc, "--", label="val")
ax.axhline(1.0, color="grey", linestyle=":", linewidth=1, label="1 bit")
ax.set(xlabel="Step", ylabel="BPC", title="Bits per character (↓ better)")
ax.legend(); ax.grid(alpha=0.3)

ax = axes[2]
ax.plot(steps, lrs, color="darkorange")
ax.set(xlabel="Step", ylabel="LR", title="LR schedule")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1e"))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved training_curves.png")


# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7 — Generate poetry from the trained model
# ═══════════════════════════════════════════════════════════════════════════════
sample = generate_sample(model, tokenizer, device,
                         n_chars=500, temperature=0.8, top_k=40)
print(sample)

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : Tesla T4


Saving corpus_small_clean.txt to corpus_small_clean.txt
Corpus uploaded: corpus_small_clean.txt
All classes and functions defined ✓
Corpus  : 1,034,622 chars | vocab: 84
Train   : 931,159 chars (930,903 samples)
Val     : 103,463 chars (103,207 samples)

Device     : cuda
Parameters : 4,812,288
Random BPC : 6.39  (baseline to beat)


W0331 16:43:24.803000 804 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


step     0 | train 4.5088 (6.505 bpc) | val 4.5060 (6.501 bpc) | lr 0.00e+00 | 47s  [patience 0/15]
          ✓ New best val loss — checkpoint saved


KeyboardInterrupt: 